# Baseline CNN — Cat Sound Classification

A from-scratch PyTorch CNN trained on log-mel spectrograms of cat vocalizations,
using the shared `kitty3000_ml` preprocessing pipeline and the fixed
`data/manifest.csv` train/val/test split. This is a first-pass baseline meant
to be iterated on, not a final model.

Spectrograms are saved into a cache directory inside data/ so taht kernel restarts do not need to rebuild them.


## Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix

import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, Audio

import librosa

from kitty3000_ml.labels import LABELS
from kitty3000_ml.preprocess import load_clip, logmel, SR, SECONDS, N_MELS

torch.manual_seed(42)

device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using device: {device}")


## Data Importing & Preprocessing

In [ ]:
DATA_DIR = "../data/raw/CatSound_originals"

manifest = pd.read_csv("../data/manifest.csv")

train_df = manifest[manifest["split"] == "train"].reset_index(drop=True)
val_df = manifest[manifest["split"] == "val"].reset_index(drop=True)
test_df = manifest[manifest["split"] == "test"].reset_index(drop=True)

print(f"train: {len(train_df)}  val: {len(val_df)}  test: {len(test_df)}")

LABEL_TO_IDX = {name: i for i, name in enumerate(LABELS)}
IDX_TO_LABEL = {v: k for k, v in LABEL_TO_IDX.items()}


## Baseline Model Dataset Class

In [ ]:
# Superseded by the forward-looping variant below ("Dataset Testing | Forward Looping").
# Kept commented out (not deleted) as the baseline reference this experiment is compared against.

# class CatSoundSpectrogramDataset(Dataset):
#     """Wraps a manifest split as (log-mel spectrogram tensor, label index) pairs.
#     Spectrograms are cached both in memory and on disk under cache_dir, so a
#     kernel restart doesn't require re-decoding every mp3 from scratch."""
#
#     def __init__(self, df, data_dir=DATA_DIR, cache_dir="../data/cache"):
#         self.df = df.reset_index(drop=True)
#         self.data_dir = data_dir
#         self.cache_dir = cache_dir
#         self._cache = {}
#
#     def __len__(self):
#         return len(self.df)
#
#     def _cache_path(self, idx):
#         row = self.df.iloc[idx]
#         rel_path = os.path.splitext(row["path"])[0] + ".npy"
#         return os.path.join(self.cache_dir, rel_path)
#
#     def _load_mel(self, idx):
#         if idx in self._cache:
#             return self._cache[idx]
#
#         cache_path = self._cache_path(idx)
#         if os.path.exists(cache_path):
#             mel = np.load(cache_path)
#         else:
#             row = self.df.iloc[idx]
#             path = os.path.join(self.data_dir, row["path"])
#             y = load_clip(path)
#             mel = logmel(y)
#             os.makedirs(os.path.dirname(cache_path), exist_ok=True)
#             np.save(cache_path, mel)
#
#         self._cache[idx] = mel
#         return mel
#
#     def __getitem__(self, idx):
#         mel = self._load_mel(idx)
#         mel = (mel - mel.mean()) / (mel.std() + 1e-6)
#         x = torch.from_numpy(mel).unsqueeze(0).float()  # shape (1, N_MELS, frames)
#         label = LABEL_TO_IDX[self.df.iloc[idx]["label"]]
#         return x, label
#
#     def preload(self, progress_widget=None):
#         for i in range(len(self)):
#             self._load_mel(i)
#             if progress_widget is not None:
#                 progress_widget.value = i + 1


## Dataset Testing | Forward Looping

Same dataset as above, except clips shorter than the fixed 11s window (`SECONDS`, staying fixed per team decision) are filled by forward-looping the waveform with a short crossfaded seam, instead of zero-padding the remainder with silence. Window size is unchanged — only how the leftover space in a short clip gets filled.

Uses its own on-disk cache directory (`../data/cache_looped`) so it never reads back stale zero-padded spectrograms from the baseline's cache.

In [ ]:
# Superseded by the audio-chunking variant below ("Dataset Testing | Audio Chunking").
# Kept commented out (not deleted) as the forward-looping reference this experiment is compared against.

# from kitty3000_ml.preprocess import N_FFT, HOP_LENGTH
#
#
# def _loop_fill(y, window, crossfade_ms=25, sr=SR):
#     """Fills `window` by forward-looping y, crossfading each seam so the splice
#     is inaudible instead of leaving a hard discontinuity (or, worse, silence)."""
#     crossfade_len = min(int(sr * crossfade_ms / 1000), len(y) // 2)
#
#     if crossfade_len <= 0:
#         # too short to crossfade meaningfully - fall back to plain tiling
#         n_repeats = int(np.ceil(window / len(y)))
#         return np.tile(y, n_repeats)[:window]
#
#     fade_out = np.linspace(1.0, 0.0, crossfade_len)
#     fade_in = np.linspace(0.0, 1.0, crossfade_len)
#
#     out = y.copy()
#     while len(out) < window:
#         # blend the tail of what we've built so far with the head of the next
#         # forward repetition of y, so the loop point has no audible click
#         tail = out[-crossfade_len:] * fade_out
#         head = y[:crossfade_len] * fade_in
#         out = np.concatenate([out[:-crossfade_len], tail + head, y[crossfade_len:]])
#
#     return out[:window]
#
#
# def load_clip_looped(path, sr=SR, seconds=SECONDS):
#     """Same as kitty3000_ml.preprocess.load_clip, except clips shorter than the
#     window are filled with a forward loop + crossfade (_loop_fill) instead of
#     zero-padding. Window is still centered on the loudest RMS frame."""
#     y, sr = librosa.load(path, sr=sr, mono=True)
#     sr = int(sr)
#
#     window = int(seconds * sr)
#
#     rms = librosa.feature.rms(y=y, frame_length=N_FFT, hop_length=HOP_LENGTH)[0]
#     loudest_frame = int(np.argmax(rms))
#     center = loudest_frame * HOP_LENGTH
#
#     start = center - window // 2
#     if start < 0:
#         start = 0
#     if start + window > len(y):
#         start = len(y) - window
#     if start < 0:
#         start = 0
#     y = y[start:start + window]
#
#     if len(y) < window:
#         y = _loop_fill(y, window, sr=sr)
#
#     return y
#
#
# def spec_augment(mel, freq_mask_param=5, time_mask_param=10, num_freq_masks=1, num_time_masks=1):
#     """Zeroes random frequency bands and time spans, standard SpecAugment-style
#     regularization. Mask value is 0 since mel is already z-score normalized
#     (mean ~0), so masked regions read as 'average', not silence or noise.
#
#     freq/time_mask_param halved from an initial 10/20 pass: that setting made
#     every training epoch harder and, on this small (2058-clip) dataset within a
#     fixed 20-epoch budget, left val/test macro F1 below the no-augmentation
#     baseline instead of above it (0.805 vs 0.824 test macro F1) even though it
#     did fix the LR-driven late-training oscillation. Lighter masking trades some
#     regularization strength for less signal loss per sample."""
#     mel = mel.copy()
#     n_mels, n_frames = mel.shape
#     for _ in range(num_freq_masks):
#         f = np.random.randint(0, freq_mask_param + 1)
#         f0 = np.random.randint(0, max(1, n_mels - f))
#         mel[f0:f0 + f, :] = 0
#     for _ in range(num_time_masks):
#         t = np.random.randint(0, time_mask_param + 1)
#         t0 = np.random.randint(0, max(1, n_frames - t))
#         mel[:, t0:t0 + t] = 0
#     return mel
#
#
# class CatSoundSpectrogramDatasetLooped(Dataset):
#     """Same as CatSoundSpectrogramDataset, but short clips are filled via
#     load_clip_looped (forward loop + crossfade) instead of zero-padding.
#
#     If augment=True, a fresh SpecAugment mask (spec_augment) is applied to the
#     normalized mel on every __getitem__ call — only the cached raw mel is shared
#     across epochs, so each epoch sees different masks. Use augment=True only for
#     the training split; val/test should stay deterministic."""
#
#     def __init__(self, df, data_dir=DATA_DIR, cache_dir="../data/cache_looped", augment=False):
#         self.df = df.reset_index(drop=True)
#         self.data_dir = data_dir
#         self.cache_dir = cache_dir
#         self.augment = augment
#         self._cache = {}
#
#     def __len__(self):
#         return len(self.df)
#
#     def _cache_path(self, idx):
#         row = self.df.iloc[idx]
#         rel_path = os.path.splitext(row["path"])[0] + ".npy"
#         return os.path.join(self.cache_dir, rel_path)
#
#     def _load_mel(self, idx):
#         if idx in self._cache:
#             return self._cache[idx]
#
#         cache_path = self._cache_path(idx)
#         if os.path.exists(cache_path):
#             mel = np.load(cache_path)
#         else:
#             row = self.df.iloc[idx]
#             path = os.path.join(self.data_dir, row["path"])
#             y = load_clip_looped(path)
#             mel = logmel(y)
#             os.makedirs(os.path.dirname(cache_path), exist_ok=True)
#             np.save(cache_path, mel)
#
#         self._cache[idx] = mel
#         return mel
#
#     def __getitem__(self, idx):
#         mel = self._load_mel(idx)
#         mel = (mel - mel.mean()) / (mel.std() + 1e-6)
#         if self.augment:
#             mel = spec_augment(mel)
#         x = torch.from_numpy(mel).unsqueeze(0).float()  # shape (1, N_MELS, frames)
#         label = LABEL_TO_IDX[self.df.iloc[idx]["label"]]
#         return x, label
#
#     def preload(self, progress_widget=None):
#         for i in range(len(self)):
#             self._load_mel(i)
#             if progress_widget is not None:
#                 progress_widget.value = i + 1

## Dataset Testing | Audio Chunking

Same underlying problem as the forward-looping variant above — this dataset's clips are mostly much shorter than any fixed window we'd pick (median ~3.8s) — but a different fix. Instead of forcing every clip into a single fixed window (looping short clips to fill it, cropping long ones down to their loudest moment and discarding the rest), each **training** clip is split into consecutive, non-overlapping `CHUNK_SECONDS`-long chunks, and every chunk becomes its own training example with its parent clip's label.

This doesn't remove the need for looping — it narrows where it applies. A clip shorter than `CHUNK_SECONDS` (still the majority of this dataset at 4s chunks) gets exactly one chunk, filled via the same crossfaded loop-fill used above. A longer clip's trailing remainder (whatever's left after slicing out as many full `CHUNK_SECONDS` chunks as fit) is loop-filled the same way rather than dropped, so no audio content is discarded. A clip long enough to yield more than one chunk contributes several real, non-looped training examples instead of just one, so this changes the training set's effective size, not just how each example is built.

Val/test stay **one example per file**, like every earlier dataset in this notebook — `CatSoundSpectrogramDatasetSingleChunk` builds that one example the same way the forward-loop dataset did (a single `CHUNK_SECONDS`-wide window, centered on the loudest RMS frame, loop-filled if the clip's shorter than the window), just at chunk width instead of the full 11s. That keeps evaluation directly comparable to every run already logged in the results table, and leaves the rest of the dashboard/sample-browser code below untouched — only training benefits from chunking's multiple examples per longer clip.

Chunks are always built from each clip's own pre-assigned split (`train_df`/`val_df`/`test_df`), so multiple chunks from the same recording can never cross a train/val/test boundary. Uses its own on-disk cache directories (`../data/cache_chunked` for train, `../data/cache_chunked_eval` for val/test), keyed per chunk rather than per clip.

In [ ]:
# Superseded by the no-loop variant below ("Dataset Testing | Audio Chunking, No Loop").
# Kept commented out (not deleted) as the loop-fill reference this experiment is compared against.

# from kitty3000_ml.preprocess import N_FFT, HOP_LENGTH
#
# CHUNK_SECONDS = 4.0
#
#
# def _loop_fill(y, window, crossfade_ms=25, sr=SR):
#     """Fills `window` by forward-looping y, crossfading each seam so the splice
#     is inaudible instead of leaving a hard discontinuity (or, worse, silence).
#     Same helper as the forward-looping section above, redefined here so this
#     section stays self-contained now that section is commented out."""
#     crossfade_len = min(int(sr * crossfade_ms / 1000), len(y) // 2)
#
#     if crossfade_len <= 0:
#         # too short to crossfade meaningfully - fall back to plain tiling
#         n_repeats = int(np.ceil(window / len(y)))
#         return np.tile(y, n_repeats)[:window]
#
#     fade_out = np.linspace(1.0, 0.0, crossfade_len)
#     fade_in = np.linspace(0.0, 1.0, crossfade_len)
#
#     out = y.copy()
#     while len(out) < window:
#         # blend the tail of what we've built so far with the head of the next
#         # forward repetition of y, so the loop point has no audible click
#         tail = out[-crossfade_len:] * fade_out
#         head = y[:crossfade_len] * fade_in
#         out = np.concatenate([out[:-crossfade_len], tail + head, y[crossfade_len:]])
#
#     return out[:window]
#
#
# def chunk_clip(path, sr=SR, chunk_seconds=CHUNK_SECONDS):
#     """Splits a clip into consecutive, non-overlapping chunk_seconds windows,
#     covering the whole clip in order instead of centering one big window on
#     the loudest moment. The trailing remainder (or the whole clip, if it's
#     shorter than one chunk) is filled up to a full window with _loop_fill, so
#     every returned chunk is exactly chunk_seconds long, but only the fill
#     portion (if any) of any given chunk is looped content. Used for training
#     only - see load_clip_single_chunk below for val/test."""
#     y, sr = librosa.load(path, sr=sr, mono=True)
#     sr = int(sr)
#     window = int(chunk_seconds * sr)
#
#     n_full_chunks = len(y) // window
#     chunks = [y[i * window:(i + 1) * window] for i in range(n_full_chunks)]
#
#     remainder = y[n_full_chunks * window:]
#     if len(remainder) > 0:
#         chunks.append(_loop_fill(remainder, window, sr=sr))
#
#     return chunks
#
#
# def load_clip_single_chunk(path, sr=SR, chunk_seconds=CHUNK_SECONDS):
#     """One chunk_seconds-wide window per file, centered on the loudest RMS
#     frame and loop-filled if the clip is shorter than the window - the same
#     approach as load_clip_looped in the forward-looping section above, just at
#     chunk_seconds width instead of the full SECONDS window. Used for val/test
#     so evaluation stays one example per file: directly comparable to every
#     earlier run in the results table, while training (chunk_clip above)
#     benefits from multiple chunks on clips long enough to have them."""
#     y, sr = librosa.load(path, sr=sr, mono=True)
#     sr = int(sr)
#
#     window = int(chunk_seconds * sr)
#
#     rms = librosa.feature.rms(y=y, frame_length=N_FFT, hop_length=HOP_LENGTH)[0]
#     loudest_frame = int(np.argmax(rms))
#     center = loudest_frame * HOP_LENGTH
#
#     start = center - window // 2
#     if start < 0:
#         start = 0
#     if start + window > len(y):
#         start = len(y) - window
#     if start < 0:
#         start = 0
#     y = y[start:start + window]
#
#     if len(y) < window:
#         y = _loop_fill(y, window, sr=sr)
#
#     return y
#
#
# class CatSoundSpectrogramDatasetChunked(Dataset):
#     """Training dataset: each clip contributes one example per chunk_clip()
#     window instead of exactly one example per clip - a clip longer than
#     chunk_seconds yields several real (non-looped) chunks, while a clip
#     shorter than chunk_seconds still yields exactly one, loop-filled.
#
#     Building the index up front (one librosa.get_duration call per row, which
#     reads duration without decoding the full clip) is what lets __len__ - and
#     therefore the preload() progress bar - report the true chunk count before
#     any audio is actually loaded."""
#
#     def __init__(self, df, data_dir=DATA_DIR, cache_dir="../data/cache_chunked", chunk_seconds=CHUNK_SECONDS):
#         self.df = df.reset_index(drop=True)
#         self.data_dir = data_dir
#         self.cache_dir = cache_dir
#         self.chunk_seconds = chunk_seconds
#         self._cache = {}
#
#         # one (row_idx, chunk_idx) entry per training example this dataset yields;
#         # built from this df alone, so chunks never cross the train/val/test split
#         # already encoded in which df (train_df/val_df/test_df) was passed in
#         self._index = []
#         for row_idx, row in self.df.iterrows():
#             duration = librosa.get_duration(path=os.path.join(self.data_dir, row["path"]))
#             n_chunks = max(1, int(np.ceil(duration / chunk_seconds)))
#             self._index.extend((row_idx, chunk_idx) for chunk_idx in range(n_chunks))
#
#     def __len__(self):
#         return len(self._index)
#
#     def _chunk_cache_path(self, row_idx, chunk_idx):
#         row = self.df.iloc[row_idx]
#         rel_path = os.path.splitext(row["path"])[0] + f"_chunk{chunk_idx}.npy"
#         return os.path.join(self.cache_dir, rel_path)
#
#     def _compute_and_cache_row_chunks(self, row_idx):
#         """Decodes a clip once and caches every one of its chunks in a single
#         pass, so requesting chunk 1 right after chunk 0 doesn't re-decode the
#         same file - decoding audio is the expensive part, not slicing it."""
#         row = self.df.iloc[row_idx]
#         path = os.path.join(self.data_dir, row["path"])
#         chunks = chunk_clip(path, chunk_seconds=self.chunk_seconds)
#
#         mels = []
#         for chunk_idx, chunk in enumerate(chunks):
#             mel = logmel(chunk)
#             cache_path = self._chunk_cache_path(row_idx, chunk_idx)
#             os.makedirs(os.path.dirname(cache_path), exist_ok=True)
#             np.save(cache_path, mel)
#             self._cache[(row_idx, chunk_idx)] = mel
#             mels.append(mel)
#         return mels
#
#     def _load_mel(self, row_idx, chunk_idx):
#         key = (row_idx, chunk_idx)
#         if key in self._cache:
#             return self._cache[key]
#
#         cache_path = self._chunk_cache_path(row_idx, chunk_idx)
#         if os.path.exists(cache_path):
#             mel = np.load(cache_path)
#             self._cache[key] = mel
#             return mel
#
#         mels = self._compute_and_cache_row_chunks(row_idx)
#         # librosa.get_duration (used to size self._index in __init__) reads
#         # duration from the mp3's header and can very slightly overestimate it
#         # for some VBR-encoded files, occasionally predicting one more chunk
#         # than chunk_clip() actually produces once the audio is really decoded
#         # (e.g. a clip whose header says 4.02s but decodes to 3.997s - one
#         # chunk, not two). Clamp instead of crashing: the "missing" chunk is a
#         # rounding artifact, not real audio content being lost.
#         return mels[min(chunk_idx, len(mels) - 1)]
#
#     def __getitem__(self, idx):
#         row_idx, chunk_idx = self._index[idx]
#         mel = self._load_mel(row_idx, chunk_idx)
#         mel = (mel - mel.mean()) / (mel.std() + 1e-6)
#         x = torch.from_numpy(mel).unsqueeze(0).float()  # shape (1, N_MELS, frames)
#         label = LABEL_TO_IDX[self.df.iloc[row_idx]["label"]]
#         return x, label
#
#     def preload(self, progress_widget=None):
#         for i in range(len(self)):
#             row_idx, chunk_idx = self._index[i]
#             self._load_mel(row_idx, chunk_idx)
#             if progress_widget is not None:
#                 progress_widget.value = i + 1
#
#
# class CatSoundSpectrogramDatasetSingleChunk(Dataset):
#     """Val/test dataset: one example per file (same shape/interface as
#     CatSoundSpectrogramDatasetLooped above), built from a single
#     chunk_seconds-wide window via load_clip_single_chunk instead of chunking
#     into several. Keeps evaluation comparable in size and format to every
#     earlier run, while the model itself now trains on chunk_seconds-wide
#     inputs via CatSoundSpectrogramDatasetChunked."""
#
#     def __init__(self, df, data_dir=DATA_DIR, cache_dir="../data/cache_chunked_eval"):
#         self.df = df.reset_index(drop=True)
#         self.data_dir = data_dir
#         self.cache_dir = cache_dir
#         self._cache = {}
#
#     def __len__(self):
#         return len(self.df)
#
#     def _cache_path(self, idx):
#         row = self.df.iloc[idx]
#         rel_path = os.path.splitext(row["path"])[0] + ".npy"
#         return os.path.join(self.cache_dir, rel_path)
#
#     def _load_mel(self, idx):
#         if idx in self._cache:
#             return self._cache[idx]
#
#         cache_path = self._cache_path(idx)
#         if os.path.exists(cache_path):
#             mel = np.load(cache_path)
#         else:
#             row = self.df.iloc[idx]
#             path = os.path.join(self.data_dir, row["path"])
#             y = load_clip_single_chunk(path)
#             mel = logmel(y)
#             os.makedirs(os.path.dirname(cache_path), exist_ok=True)
#             np.save(cache_path, mel)
#
#         self._cache[idx] = mel
#         return mel
#
#     def __getitem__(self, idx):
#         mel = self._load_mel(idx)
#         mel = (mel - mel.mean()) / (mel.std() + 1e-6)
#         x = torch.from_numpy(mel).unsqueeze(0).float()  # shape (1, N_MELS, frames)
#         label = LABEL_TO_IDX[self.df.iloc[idx]["label"]]
#         return x, label
#
#     def preload(self, progress_widget=None):
#         for i in range(len(self)):
#             self._load_mel(i)
#             if progress_widget is not None:
#                 progress_widget.value = i + 1

## Dataset Testing | Audio Chunking, No Loop

Isolates one variable from the chunking section above: same `CHUNK_SECONDS`-wide windowing (multiple chunks per training clip, one loudest-centered window per val/test file), but the fallback for anything shorter than the window is **zero-padding (silence)** instead of the crossfaded loop-fill — the same fallback the very first baseline's `load_clip()` used, just at `CHUNK_SECONDS` width instead of the full 11s.

The question this answers: now that the window is small enough that most clips only need a little filling (not the ~7s of looped content nearly every clip needed at the old 11s window), is the loop-fill's crossfade still earning its keep, or did it stop mattering once there was this little left to fill? Comparing this run's score against the loop-fill chunking run above (test macro F1 0.8391) isolates exactly that.

Same split guarantee as above: chunks are built from each clip's own pre-assigned split, so nothing crosses train/val/test. Uses its own cache directories (`../data/cache_chunked_noloop` for train, `../data/cache_chunked_eval_noloop` for val/test) so it never reads back stale loop-filled spectrograms from the section above.

In [ ]:
# Superseded by the natural-length evaluation variant below ("Dataset Testing | Natural-Length Evaluation").
# Kept commented out (not deleted) as the zero-pad reference this experiment is compared against.

# from kitty3000_ml.preprocess import N_FFT, HOP_LENGTH
#
# CHUNK_SECONDS = 4.0
#
#
# def chunk_clip_no_loop(path, sr=SR, chunk_seconds=CHUNK_SECONDS):
#     """Same windowing as chunk_clip in the section above, but the trailing
#     remainder (or the whole clip, if it's shorter than one chunk) is
#     zero-padded instead of loop-filled - the same fallback
#     kitty3000_ml.preprocess.load_clip used for the original baseline, at
#     chunk_seconds width instead of the full SECONDS window."""
#     y, sr = librosa.load(path, sr=sr, mono=True)
#     sr = int(sr)
#     window = int(chunk_seconds * sr)
#
#     n_full_chunks = len(y) // window
#     chunks = [y[i * window:(i + 1) * window] for i in range(n_full_chunks)]
#
#     remainder = y[n_full_chunks * window:]
#     if len(remainder) > 0:
#         chunks.append(np.pad(remainder, (0, window - len(remainder))))
#
#     return chunks
#
#
# def load_clip_single_chunk_no_loop(path, sr=SR, chunk_seconds=CHUNK_SECONDS):
#     """Same as load_clip_single_chunk in the section above, but zero-pads a
#     too-short clip instead of loop-filling it."""
#     y, sr = librosa.load(path, sr=sr, mono=True)
#     sr = int(sr)
#
#     window = int(chunk_seconds * sr)
#
#     rms = librosa.feature.rms(y=y, frame_length=N_FFT, hop_length=HOP_LENGTH)[0]
#     loudest_frame = int(np.argmax(rms))
#     center = loudest_frame * HOP_LENGTH
#
#     start = center - window // 2
#     if start < 0:
#         start = 0
#     if start + window > len(y):
#         start = len(y) - window
#     if start < 0:
#         start = 0
#     y = y[start:start + window]
#
#     if len(y) < window:
#         y = np.pad(y, (0, window - len(y)))
#
#     return y
#
#
# class CatSoundSpectrogramDatasetChunkedNoLoop(Dataset):
#     """Same as CatSoundSpectrogramDatasetChunked in the section above, but
#     built from chunk_clip_no_loop - zero-padding instead of loop-filling short
#     clips and trailing remainders."""
#
#     def __init__(self, df, data_dir=DATA_DIR, cache_dir="../data/cache_chunked_noloop", chunk_seconds=CHUNK_SECONDS):
#         self.df = df.reset_index(drop=True)
#         self.data_dir = data_dir
#         self.cache_dir = cache_dir
#         self.chunk_seconds = chunk_seconds
#         self._cache = {}
#
#         self._index = []
#         for row_idx, row in self.df.iterrows():
#             duration = librosa.get_duration(path=os.path.join(self.data_dir, row["path"]))
#             n_chunks = max(1, int(np.ceil(duration / chunk_seconds)))
#             self._index.extend((row_idx, chunk_idx) for chunk_idx in range(n_chunks))
#
#     def __len__(self):
#         return len(self._index)
#
#     def _chunk_cache_path(self, row_idx, chunk_idx):
#         row = self.df.iloc[row_idx]
#         rel_path = os.path.splitext(row["path"])[0] + f"_chunk{chunk_idx}.npy"
#         return os.path.join(self.cache_dir, rel_path)
#
#     def _compute_and_cache_row_chunks(self, row_idx):
#         row = self.df.iloc[row_idx]
#         path = os.path.join(self.data_dir, row["path"])
#         chunks = chunk_clip_no_loop(path, chunk_seconds=self.chunk_seconds)
#
#         mels = []
#         for chunk_idx, chunk in enumerate(chunks):
#             mel = logmel(chunk)
#             cache_path = self._chunk_cache_path(row_idx, chunk_idx)
#             os.makedirs(os.path.dirname(cache_path), exist_ok=True)
#             np.save(cache_path, mel)
#             self._cache[(row_idx, chunk_idx)] = mel
#             mels.append(mel)
#         return mels
#
#     def _load_mel(self, row_idx, chunk_idx):
#         key = (row_idx, chunk_idx)
#         if key in self._cache:
#             return self._cache[key]
#
#         cache_path = self._chunk_cache_path(row_idx, chunk_idx)
#         if os.path.exists(cache_path):
#             mel = np.load(cache_path)
#             self._cache[key] = mel
#             return mel
#
#         mels = self._compute_and_cache_row_chunks(row_idx)
#         # same librosa.get_duration-vs-decoded-length rounding edge case as
#         # CatSoundSpectrogramDatasetChunked above - clamp rather than crash
#         return mels[min(chunk_idx, len(mels) - 1)]
#
#     def __getitem__(self, idx):
#         row_idx, chunk_idx = self._index[idx]
#         mel = self._load_mel(row_idx, chunk_idx)
#         mel = (mel - mel.mean()) / (mel.std() + 1e-6)
#         x = torch.from_numpy(mel).unsqueeze(0).float()  # shape (1, N_MELS, frames)
#         label = LABEL_TO_IDX[self.df.iloc[row_idx]["label"]]
#         return x, label
#
#     def preload(self, progress_widget=None):
#         for i in range(len(self)):
#             row_idx, chunk_idx = self._index[i]
#             self._load_mel(row_idx, chunk_idx)
#             if progress_widget is not None:
#                 progress_widget.value = i + 1
#
#
# class CatSoundSpectrogramDatasetSingleChunkNoLoop(Dataset):
#     """Same as CatSoundSpectrogramDatasetSingleChunk in the section above, but
#     built from load_clip_single_chunk_no_loop - zero-padding instead of
#     loop-filling."""
#
#     def __init__(self, df, data_dir=DATA_DIR, cache_dir="../data/cache_chunked_eval_noloop"):
#         self.df = df.reset_index(drop=True)
#         self.data_dir = data_dir
#         self.cache_dir = cache_dir
#         self._cache = {}
#
#     def __len__(self):
#         return len(self.df)
#
#     def _cache_path(self, idx):
#         row = self.df.iloc[idx]
#         rel_path = os.path.splitext(row["path"])[0] + ".npy"
#         return os.path.join(self.cache_dir, rel_path)
#
#     def _load_mel(self, idx):
#         if idx in self._cache:
#             return self._cache[idx]
#
#         cache_path = self._cache_path(idx)
#         if os.path.exists(cache_path):
#             mel = np.load(cache_path)
#         else:
#             row = self.df.iloc[idx]
#             path = os.path.join(self.data_dir, row["path"])
#             y = load_clip_single_chunk_no_loop(path)
#             mel = logmel(y)
#             os.makedirs(os.path.dirname(cache_path), exist_ok=True)
#             np.save(cache_path, mel)
#
#         self._cache[idx] = mel
#         return mel
#
#     def __getitem__(self, idx):
#         mel = self._load_mel(idx)
#         mel = (mel - mel.mean()) / (mel.std() + 1e-6)
#         x = torch.from_numpy(mel).unsqueeze(0).float()  # shape (1, N_MELS, frames)
#         label = LABEL_TO_IDX[self.df.iloc[idx]["label"]]
#         return x, label
#
#     def preload(self, progress_widget=None):
#         for i in range(len(self)):
#             self._load_mel(i)
#             if progress_widget is not None:
#                 progress_widget.value = i + 1

## Dataset Testing | Natural-Length Evaluation

Addresses a real gap in every dataset section above: val/test have always gone through *some* fixed-window fallback (loop-fill or zero-pad) whenever a clip is shorter than the window — which is most clips. A real user will never upload a looped or silence-padded recording, so evaluating on that isn't 100% honest about deployed performance, however close it gets.

Training keeps the best recipe found so far unchanged: `CatSoundSpectrogramDatasetChunked` below (4s chunks, loop-filled short clips/remainders) is reused as-is, since training data being synthetically expanded/manipulated is standard practice and doesn't carry the same "will a user ever see this" concern.

Val/test change fundamentally: `CatSoundSpectrogramDatasetNatural` returns each clip's own full, **unmodified** spectrogram — no fixed window, no crop, no pad, no loop, nothing synthetic. This is possible because `CNNBaseline`'s last feature layer is `nn.AdaptiveAvgPool2d((1, 1))` (see the CNN Architecture section below), which pools *any* spatial size down to a fixed 1×1 before the classifier ever sees it — the model was never architecturally constrained to a fixed window, that constraint only ever existed for batch collation. Since these tensors are now variable-shaped, `val_loader`/`test_loader` switch to `batch_size=1` in the DataLoaders section below (PyTorch's default collate can't stack differently-shaped tensors) — a free change here, since val/test are only 448/447 samples and `model.eval()` already uses BatchNorm's running stats rather than per-batch stats.

Uses its own cache directory (`../data/cache_natural`) for val/test; training keeps reading from the existing `../data/cache_chunked` cache from the loop-fill chunking run, so no retraining-cache work is repeated.

In [ ]:
CHUNK_SECONDS = 4.0


def _loop_fill(y, window, crossfade_ms=25, sr=SR):
    """Fills `window` by forward-looping y, crossfading each seam so the splice
    is inaudible instead of leaving a hard discontinuity (or, worse, silence).
    Same helper as the earlier chunking section, redefined here so this
    section stays self-contained now that section is commented out."""
    crossfade_len = min(int(sr * crossfade_ms / 1000), len(y) // 2)

    if crossfade_len <= 0:
        # too short to crossfade meaningfully - fall back to plain tiling
        n_repeats = int(np.ceil(window / len(y)))
        return np.tile(y, n_repeats)[:window]

    fade_out = np.linspace(1.0, 0.0, crossfade_len)
    fade_in = np.linspace(0.0, 1.0, crossfade_len)

    out = y.copy()
    while len(out) < window:
        # blend the tail of what we've built so far with the head of the next
        # forward repetition of y, so the loop point has no audible click
        tail = out[-crossfade_len:] * fade_out
        head = y[:crossfade_len] * fade_in
        out = np.concatenate([out[:-crossfade_len], tail + head, y[crossfade_len:]])

    return out[:window]


def chunk_clip(path, sr=SR, chunk_seconds=CHUNK_SECONDS):
    """Splits a clip into consecutive, non-overlapping chunk_seconds windows,
    covering the whole clip in order instead of centering one big window on
    the loudest moment. The trailing remainder (or the whole clip, if it's
    shorter than one chunk) is filled up to a full window with _loop_fill, so
    every returned chunk is exactly chunk_seconds long, but only the fill
    portion (if any) of any given chunk is looped content. Training only -
    val/test use CatSoundSpectrogramDatasetNatural below, with no windowing
    or fill of any kind."""
    y, sr = librosa.load(path, sr=sr, mono=True)
    sr = int(sr)
    window = int(chunk_seconds * sr)

    n_full_chunks = len(y) // window
    chunks = [y[i * window:(i + 1) * window] for i in range(n_full_chunks)]

    remainder = y[n_full_chunks * window:]
    if len(remainder) > 0:
        chunks.append(_loop_fill(remainder, window, sr=sr))

    return chunks


class CatSoundSpectrogramDatasetChunked(Dataset):
    """Training dataset (unchanged from the loop-fill chunking section above):
    each clip contributes one example per chunk_clip() window instead of
    exactly one example per clip - a clip longer than chunk_seconds yields
    several real (non-looped) chunks, while a clip shorter than chunk_seconds
    still yields exactly one, loop-filled.

    Building the index up front (one librosa.get_duration call per row, which
    reads duration without decoding the full clip) is what lets __len__ - and
    therefore the preload() progress bar - report the true chunk count before
    any audio is actually loaded."""

    def __init__(self, df, data_dir=DATA_DIR, cache_dir="../data/cache_chunked", chunk_seconds=CHUNK_SECONDS):
        self.df = df.reset_index(drop=True)
        self.data_dir = data_dir
        self.cache_dir = cache_dir
        self.chunk_seconds = chunk_seconds
        self._cache = {}

        # one (row_idx, chunk_idx) entry per training example this dataset yields;
        # built from this df alone, so chunks never cross the train/val/test split
        # already encoded in which df (train_df/val_df/test_df) was passed in
        self._index = []
        for row_idx, row in self.df.iterrows():
            duration = librosa.get_duration(path=os.path.join(self.data_dir, row["path"]))
            n_chunks = max(1, int(np.ceil(duration / chunk_seconds)))
            self._index.extend((row_idx, chunk_idx) for chunk_idx in range(n_chunks))

    def __len__(self):
        return len(self._index)

    def _chunk_cache_path(self, row_idx, chunk_idx):
        row = self.df.iloc[row_idx]
        rel_path = os.path.splitext(row["path"])[0] + f"_chunk{chunk_idx}.npy"
        return os.path.join(self.cache_dir, rel_path)

    def _compute_and_cache_row_chunks(self, row_idx):
        """Decodes a clip once and caches every one of its chunks in a single
        pass, so requesting chunk 1 right after chunk 0 doesn't re-decode the
        same file - decoding audio is the expensive part, not slicing it."""
        row = self.df.iloc[row_idx]
        path = os.path.join(self.data_dir, row["path"])
        chunks = chunk_clip(path, chunk_seconds=self.chunk_seconds)

        mels = []
        for chunk_idx, chunk in enumerate(chunks):
            mel = logmel(chunk)
            cache_path = self._chunk_cache_path(row_idx, chunk_idx)
            os.makedirs(os.path.dirname(cache_path), exist_ok=True)
            np.save(cache_path, mel)
            self._cache[(row_idx, chunk_idx)] = mel
            mels.append(mel)
        return mels

    def _load_mel(self, row_idx, chunk_idx):
        key = (row_idx, chunk_idx)
        if key in self._cache:
            return self._cache[key]

        cache_path = self._chunk_cache_path(row_idx, chunk_idx)
        if os.path.exists(cache_path):
            mel = np.load(cache_path)
            self._cache[key] = mel
            return mel

        mels = self._compute_and_cache_row_chunks(row_idx)
        # librosa.get_duration (used to size self._index in __init__) reads
        # duration from the mp3's header and can very slightly overestimate it
        # for some VBR-encoded files, occasionally predicting one more chunk
        # than chunk_clip() actually produces once the audio is really decoded.
        # Clamp instead of crashing: the "missing" chunk is a rounding
        # artifact, not real audio content being lost.
        return mels[min(chunk_idx, len(mels) - 1)]

    def __getitem__(self, idx):
        row_idx, chunk_idx = self._index[idx]
        mel = self._load_mel(row_idx, chunk_idx)
        mel = (mel - mel.mean()) / (mel.std() + 1e-6)
        x = torch.from_numpy(mel).unsqueeze(0).float()  # shape (1, N_MELS, frames)
        label = LABEL_TO_IDX[self.df.iloc[row_idx]["label"]]
        return x, label

    def preload(self, progress_widget=None):
        for i in range(len(self)):
            row_idx, chunk_idx = self._index[i]
            self._load_mel(row_idx, chunk_idx)
            if progress_widget is not None:
                progress_widget.value = i + 1


class CatSoundSpectrogramDatasetNatural(Dataset):
    """Val/test dataset: each clip's own full, natural-length spectrogram -
    no crop, no pad, no loop, no synthetic content of any kind. This is
    exactly what a deployed model receives from a real user upload, so
    metrics measured this way have 100% data integrity.

    __getitem__ returns variable-shaped tensors (frames scales with each
    clip's own duration) since there's no fixed window at all - a DataLoader
    built from this class MUST use batch_size=1 (see the DataLoaders section
    below), since the default collate can't stack differently-shaped tensors
    into one batch. That's fine here: only 448/447 samples, and CNNBaseline's
    AdaptiveAvgPool2d already makes single-sample forward passes work
    regardless of input width."""

    def __init__(self, df, data_dir=DATA_DIR, cache_dir="../data/cache_natural"):
        self.df = df.reset_index(drop=True)
        self.data_dir = data_dir
        self.cache_dir = cache_dir
        self._cache = {}

    def __len__(self):
        return len(self.df)

    def _cache_path(self, idx):
        row = self.df.iloc[idx]
        rel_path = os.path.splitext(row["path"])[0] + ".npy"
        return os.path.join(self.cache_dir, rel_path)

    def _load_mel(self, idx):
        if idx in self._cache:
            return self._cache[idx]

        cache_path = self._cache_path(idx)
        if os.path.exists(cache_path):
            mel = np.load(cache_path)
        else:
            row = self.df.iloc[idx]
            path = os.path.join(self.data_dir, row["path"])
            y, _ = librosa.load(path, sr=SR, mono=True)  # no window, no crop, no fill of any kind
            mel = logmel(y)
            os.makedirs(os.path.dirname(cache_path), exist_ok=True)
            np.save(cache_path, mel)

        self._cache[idx] = mel
        return mel

    def __getitem__(self, idx):
        mel = self._load_mel(idx)
        mel = (mel - mel.mean()) / (mel.std() + 1e-6)
        x = torch.from_numpy(mel).unsqueeze(0).float()  # shape (1, N_MELS, frames) - frames varies per clip
        label = LABEL_TO_IDX[self.df.iloc[idx]["label"]]
        return x, label

    def preload(self, progress_widget=None):
        for i in range(len(self)):
            self._load_mel(i)
            if progress_widget is not None:
                progress_widget.value = i + 1

## Train, Test, Validate Datasets


In [ ]:
train_ds = CatSoundSpectrogramDatasetChunked(train_df)
val_ds = CatSoundSpectrogramDatasetNatural(val_df)
test_ds = CatSoundSpectrogramDatasetNatural(test_df)

for name, ds in [("train", train_ds), ("val", val_ds), ("test", test_ds)]:
    progress = widgets.IntProgress(min=0, max=len(ds), description=f"{name}:")
    display(progress)
    ds.preload(progress)

## Sanity Check Spectrogram

In [ ]:
x, y = train_ds[0]
print(f"Sample tensor shape: {x.shape}, dtype: {x.dtype}, label: {IDX_TO_LABEL[y]}")

plt.figure(figsize=(6, 3))
plt.imshow(x[0].numpy(), origin="lower", aspect="auto")
plt.title(f"Example spectrogram — {IDX_TO_LABEL[y]}")
plt.xlabel("time frame")
plt.ylabel("mel bin")
plt.colorbar()
plt.show()

## DataLoaders

Groupg individual examples into batches for training. Only the training loader shuffles: val/test order doesn't matter and we want it stable so we can line predictions back up with val_df/test_df rows later.

In [ ]:
BATCH_SIZE = 32

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
# val/test now yield each clip's own natural, variable-length spectrogram (no
# fixed window at all) - the default collate can't stack differently-shaped
# tensors into a batch, so these use batch_size=1. No real cost here: only
# 448/447 samples, and model.eval() already uses BatchNorm's running stats
# rather than per-batch stats, so batch_size=1 isn't a training-time concern.
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=0)


## CNN Architecture

In [ ]:
class CNNBaseline(nn.Module):
    def __init__(self, n_classes=len(LABELS), dropout=0.3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1), nn.BatchNorm2d(16), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(128, 64), nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(64, n_classes),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


## Instantiate Model

In [ ]:
model = CNNBaseline().to(device)
n_params = sum(p.numel() for p in model.parameters())

print(f"Model has {n_params:,} parameters")

## Loss, Optimizer & Hyperparameters

In [ ]:
N_EPOCHS = 20
LR = 1e-3

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)

# Tracks val_macro_f1 (the same metric already used for checkpoint selection below) and
# halves LR whenever it stalls for `patience` epochs, to stabilize the late-training
# oscillation seen with a constant 1e-3 LR.
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=3, min_lr=1e-5
)

os.makedirs("../models/cnn_baseline", exist_ok=True)
CHECKPOINT_PATH = "../models/cnn_baseline/cnn_baseline_best.pt"

## Training

In [ ]:
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "val_macro_f1": [], "lr": []}
best_val_f1 = -1.0

for epoch in range(N_EPOCHS):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * x.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        total += x.size(0)

    train_loss = running_loss / total
    train_acc = correct / total

    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    y_true, y_pred = [], []
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = criterion(logits, y)
            val_loss += loss.item() * x.size(0)
            preds = logits.argmax(1)
            val_correct += (preds == y).sum().item()
            val_total += x.size(0)
            y_true.extend(y.cpu().tolist())
            y_pred.extend(preds.cpu().tolist())

    val_loss /= val_total
    val_acc = val_correct / val_total
    val_macro_f1 = f1_score(y_true, y_pred, average="macro")

    # steps the LR down when val_macro_f1 stalls; must run after val_macro_f1 is known
    # and before checkpointing so a just-lowered LR is reflected in this epoch's log line
    scheduler.step(val_macro_f1)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    history["val_macro_f1"].append(val_macro_f1)
    history["lr"].append(optimizer.param_groups[0]["lr"])

    print(f"epoch {epoch+1}/{N_EPOCHS}  train_loss={train_loss:.3f}  "
          f"val_loss={val_loss:.3f}  val_acc={val_acc:.3f}  val_macro_f1={val_macro_f1:.3f}  "
          f"lr={optimizer.param_groups[0]['lr']:.2e}")

    if val_macro_f1 > best_val_f1:
        best_val_f1 = val_macro_f1
        torch.save(
            {"state_dict": model.state_dict(), "labels": LABELS,
             "val_macro_f1": val_macro_f1, "epoch": epoch},
            CHECKPOINT_PATH,
        )


print(f"Best val macro F1: {best_val_f1:.4f}")

## Testing / Verification

In [ ]:
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
model.load_state_dict(checkpoint["state_dict"])
model.eval()

y_true, y_pred, y_prob = [], [], []
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        logits = model(x)
        probs = torch.softmax(logits, dim=1)
        preds = probs.argmax(1)
        y_true.extend(y.tolist())
        y_pred.extend(preds.cpu().tolist())
        y_prob.extend(probs.cpu().tolist())

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_prob = np.array(y_prob)


## Metrics

In [ ]:
from typing import cast

# cast: classification_report's stub doesn't narrow to dict from a plain bool
# output_dict=True, so Pylance infers the str-returning overload instead and
# flags .pop()/string indexing below as invalid - a stub limitation, not a
# runtime issue (output_dict=True always returns a dict at runtime).
report = cast(dict, classification_report(y_true, y_pred, target_names=LABELS, output_dict=True, zero_division=0))
accuracy = report.pop("accuracy")
macro_f1 = report["macro avg"]["f1-score"]
weighted_f1 = report["weighted avg"]["f1-score"]

per_class_df = pd.DataFrame(report).T.drop(index=["macro avg", "weighted avg"]).sort_values("f1-score")
weakest = per_class_df.head(3)

print(f"Test accuracy:    {accuracy:.4f}")
print(f"Test macro F1:    {macro_f1:.4f}  <- headline metric (unweighted mean across classes)")
print(f"Test weighted F1: {weighted_f1:.4f}  (weighted by class support; close to macro since classes are near-balanced)")
print("Weakest classes:  " + ", ".join(f"{cls} ({row['f1-score']:.2f})" for cls, row in weakest.iterrows()))

# sorted weakest-first so the classes needing attention are the first thing you see;
# background_gradient is static HTML (no ipywidgets), so it still renders in a plain
# GitHub/nbviewer preview, unlike the interactive dashboard below
display(
    per_class_df.style
    .background_gradient(subset=["precision", "recall", "f1-score"], cmap="RdYlGn", vmin=0, vmax=1)
    .format({"precision": "{:.2f}", "recall": "{:.2f}", "f1-score": "{:.2f}", "support": "{:.0f}"})
    .set_caption("Per-class metrics, sorted weakest first")
)

### Baseline Model Metrics (historical reference)



Log a row here each time a baseline/fine-tuned variant finishes a run, using the values printed in [Metrics](#Metrics) above (`accuracy`, `macro_f1`, `weighted_f1`, `report["macro avg"]`, `weakest`) plus `N_EPOCHS` and `n_params`. Keeps a fixed reference point to compare new runs against as configs change.

| Date | Model / Config | Epochs | Params | Test Accuracy | Macro F1 | Weighted F1 | Macro Precision | Macro Recall | Weakest Classes (F1) | Notes |
|---|---|---|---|---|---|---|---|---|---|---|
| _YYYY-MM-DD_ | _e.g. baseline v1 (dropout=0.3, lr=1e-3)_ | _e.g. 20_ | _e.g. 106,538_ | _0.XX_ | _0.XX_ | _0.XX_ | _0.XX_ | _0.XX_ | _ClassA (0.XX), ClassB (0.XX)_ | _notes_ |
| 2026-09-10 | baseline v1 (dropout=0.3, lr=1e-3) | 20 | 106,538 | 0.6890 | 0.6954 | 0.6954 | ~0.74 | ~0.69 | Happy (0.53), Paining (0.53), Mating (0.56) | from last executed run (cell `341c77b1`); macro precision/recall derived from the displayed 2-decimal per-class values, so approximate |
| 2026-09-10 | forward-loop dataset (crossfade fill, dropout=0.3, lr=1e-3) | 20 | 106,538 | 0.8210 | 0.8192 | 0.8194 | 0.827 | 0.820 | Paining (0.62), Happy (0.73), Warning (0.78) | `CatSoundSpectrogramDatasetLooped` replacing zero-padding with a forward loop + 25ms crossfade; accumulated readings from this run vs. baseline v1 above: accuracy +0.132, macro F1 +0.124 |
| 2026-09-10 | LR scheduler (ReduceLROnPlateau) + SpecAugment, freq=10/time=20 (dropout=0.3, lr=1e-3→5e-4→2.5e-4) | 20 | 106,538 | 0.8076 | 0.8052 | 0.8052 | 0.810 | 0.807 | Paining (0.57), Happy (0.66), Mating (0.73) | val_macro_f1 in the last 5 epochs settled to 0.79–0.81 (vs. wild 0.665–0.829 swings in the prior run) — scheduler fixed the late-training instability, but the peak/test score is slightly below the forward-loop run above, likely because SpecAugment makes each epoch harder and 20 epochs wasn't enough to fully recover; worth trying more epochs or lighter masking (see notebook discussion) |
| 2026-09-10 | LR scheduler (ReduceLROnPlateau) + SpecAugment, freq=5/time=10 (dropout=0.3, lr=1e-3→5e-4→2.5e-4) | 20 | 106,538 | 0.8188 | 0.8175 | 0.8175 | 0.819 | 0.819 | Paining (0.65), Happy (0.67), Warning (0.76) | halved the mask sizes from the row above; recovered almost all of the regression and now roughly matches the forward-loop run (accuracy -0.002, macro F1 -0.002) while keeping the scheduler's stabilized late-training curve (best val macro F1 0.8336, the best of any run so far) — lighter SpecAugment + LR scheduling together look like a small net win over forward-loop alone |
| 2026-09-10 | Audio chunking, CHUNK_SECONDS=4.0, loop-fill, **windowed val/test** (train: `CatSoundSpectrogramDatasetChunked`, val/test: `CatSoundSpectrogramDatasetSingleChunk`; dropout=0.3, lr=1e-3→5e-4, LR scheduler on, no SpecAugment) | 20 | 106,538 | 0.8389 | 0.8391 | 0.8391 | 0.843 | 0.839 | Paining (0.71), Happy (0.76), Angry (0.77) | Best-scoring run so far, but val/test still used a synthetic loudest-centered 4s window (loop-filled when short) — not what a real user's upload looks like. Superseded for reporting purposes by the natural-length eval run below; kept as the record of the ceiling this windowing approach could reach |
| 2026-09-10 | Audio chunking, CHUNK_SECONDS=4.0, **no loop** / zero-pad fill, windowed val/test (train: `CatSoundSpectrogramDatasetChunkedNoLoop`, val/test: `CatSoundSpectrogramDatasetSingleChunkNoLoop`; dropout=0.3, lr=1e-3→5e-4, LR scheduler on, no SpecAugment) | 20 | 106,538 | 0.7830 | 0.7834 | 0.7834 | 0.798 | 0.783 | Paining (0.60), Mating (0.68), Happy (0.69) | Same 2996-chunk train set/window/schedule as the loop-fill run above, only the fill strategy changed (silence instead of crossfaded loop). Clearly worse across the board: -0.056 accuracy, -0.056 macro F1 vs. the loop-fill run, and train_loss stayed markedly higher all 20 epochs (0.740 final vs. 0.513 for loop-fill) - the model visibly struggled more to fit silence-padded inputs. Answers the "does looping still matter at 4s?" question: yes, clearly - looping is not just a vestige of the old 11s window, it's actively still worth ~5-6 points at 4s too |
| 2026-09-10 | Audio chunking train (loop-fill, unchanged) + **natural-length, no-window val/test** (train: `CatSoundSpectrogramDatasetChunked`, val/test: `CatSoundSpectrogramDatasetNatural`, batch_size=1; dropout=0.3, lr=1e-3→5e-4, LR scheduler on, no SpecAugment) | 20 | 106,538 | **0.8210** | **0.8211** | 0.8211 | 0.832 | 0.821 | Paining (0.69), Happy (0.73), Angry (0.75) | **First 100%-honest evaluation**: val/test use each clip's own real, unmodified audio - no crop, no pad, no loop, nothing synthetic (`nn.AdaptiveAvgPool2d((1,1))` means the CNN never actually required a fixed window; that constraint only existed for batch collation, so val/test now run at batch_size=1 instead). 1.8 points below the windowed-eval run above, which makes sense: that run benefited from evaluating on the same loudest-centered, length-normalized distribution the model trained on. This 0.8211 is the trustworthy number going forward - very close to the original forward-loop run's 0.8192, meaning most of the apparent gains from chunking/window-size tuning were partly an artifact of val/test also getting easier, not just the model getting better. Should be the baseline for judging any future change from here on |


## Analytics Dashboard

In [ ]:
from typing import cast

from kitty3000_ml.cnn_analytics import (
    build_training_curves_panel, build_confusion_matrix_panel, build_per_class_panel,
    build_sample_browser_panel, build_duration_bias_panel, build_umap_panel, build_pca_panel,
    GradCAM, build_saliency_panel, build_analytics_grid, close_widget_tree,
)

def extract_features(model, loader):
    model.eval()
    feats, labels = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            feats.append(model.features(x).flatten(1).cpu().numpy())
            labels.extend(y.tolist())
    return np.concatenate(feats), np.array(labels)

test_features, test_labels = extract_features(model, test_loader)
durations = [librosa.get_duration(path=os.path.join(DATA_DIR, p)) for p in test_df["path"]]

# tear down the previous run's hooks before registering a new pair on the same
# layer, so re-running this cell doesn't stack hooks on the model forever
old_grad_cam = globals().get("grad_cam")
if old_grad_cam is not None:
    old_grad_cam.remove_hooks()

# required: full_backward_hook conflicts with in-place ReLU. Cast to nn.ReLU since
# nn.Sequential.__getitem__ is typed as returning the generic nn.Module, which
# doesn't statically expose `.inplace` (Pylance false positive, not a runtime issue).
cast(nn.ReLU, model.features[-2]).inplace = False
grad_cam = GradCAM(model, model.features[-2])

def to_input_tensor(mel):
    return torch.from_numpy(mel).unsqueeze(0).unsqueeze(0).float().to(device)

# only set on the very first run - a real grid built by the cell below must survive
# subsequent re-runs of *this* cell so that cell can still close it before rebuilding
if "grid" not in globals():
    grid = None

In [ ]:
close_widget_tree(grid)  # tear down the previous run's grid before displaying a fresh one, so it doesn't linger and double up

grid = build_analytics_grid([
    (0, 0, build_training_curves_panel(history)),
    (0, 1, build_confusion_matrix_panel(y_true, y_pred, LABELS)),
    (1, 0, build_per_class_panel(y_true, y_pred, LABELS)),
    (1, 1, build_duration_bias_panel(y_true, y_pred, durations, LABELS)),
    (2, 0, build_umap_panel(test_features, y_true, y_pred, LABELS)),
    (2, 1, build_pca_panel(test_features, y_true, y_pred, LABELS)),
    (3, 0, build_saliency_panel(
        grad_cam, LABELS, y_true, y_pred,
        get_normalized_mel=lambda i: (lambda m: (m - m.mean()) / (m.std() + 1e-6))(test_ds._load_mel(i)),
        to_input_tensor=to_input_tensor, num_samples=len(test_df))),
])
display(grid)

## Sample Inspector

Pulled out of the grid above into its own full-width section — it's the main tool
for browsing individual test clips (by class or misclassifications only), so it
needs more room than a single grid cell for its waveform, spectrogram, probability
bars, and audio player to all be legible at once.

In [ ]:
sample_browser = build_sample_browser_panel(
    LABELS, y_true, y_pred, y_prob,
    # the clip's real, unmodified audio - matches what test_ds actually evaluated on,
    # not the old fixed-11s-window load_clip() which no longer reflects the eval pipeline
    get_waveform=lambda i: librosa.load(os.path.join(DATA_DIR, test_df.iloc[i]["path"]), sr=SR, mono=True)[0],
    get_mel=lambda i: test_ds._load_mel(i),
    sample_rate=SR, num_samples=len(test_df),
)
sample_browser.layout.width = "100%"
sample_browser.layout.border = "1px solid rgba(128, 128, 128, 0.4)"
sample_browser.layout.padding = "8px"
display(sample_browser)